# Schedule Job Control v1

目标：验证 `src/schedule` 的通用 job 控制契约，确保 disabled job 不执行 task、不创建 lock、不发送成功/失败通知；enabled 与 locked 行为保持可审阅。

In [1]:
from pathlib import Path
import json
import sys
import tempfile

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ScheduleJob, run_job_once

OUTPUT_DIR = REPO_ROOT / "research" / "schedule_job_control_v1"
events: list[str] = []
calls = {"enabled": 0, "disabled": 0, "locked": 0}

with tempfile.TemporaryDirectory() as raw_tmp:
    lock_dir = Path(raw_tmp) / "locks"

    enabled_job = ScheduleJob(name="dashboard_export_enabled", task="refresh_dashboard", cron="manual")
    enabled_result = run_job_once(
        enabled_job,
        task=lambda: calls.__setitem__("enabled", calls["enabled"] + 1) or {"rows": 32},
        lock_dir=lock_dir,
        notify=events.append,
    )

    disabled_job = ScheduleJob(
        name="dashboard_export_disabled",
        task="refresh_dashboard",
        cron="manual",
        enabled=False,
    )
    disabled_result = run_job_once(
        disabled_job,
        task=lambda: calls.__setitem__("disabled", calls["disabled"] + 1) or {"should_not_run": True},
        lock_dir=lock_dir,
        notify=events.append,
    )

    locked_name = "dashboard_export_locked"
    lock_dir.mkdir(parents=True, exist_ok=True)
    (lock_dir / f"{locked_name}.lock").write_text(json.dumps({"owner": "notebook"}), encoding="utf-8")
    locked_job = ScheduleJob(name=locked_name, task="refresh_dashboard", cron="manual")
    locked_result = run_job_once(
        locked_job,
        task=lambda: calls.__setitem__("locked", calls["locked"] + 1) or {"should_not_run": True},
        lock_dir=lock_dir,
        notify=events.append,
    )
    disabled_lock_exists = (lock_dir / "dashboard_export_disabled.lock").exists()

summary = {
    "schema_version": "research.schedule_job_control.v1",
    "status": "completed",
    "job_control_contract": "disabled jobs skip before lock/task/notify; locked jobs skip existing concurrent work; enabled jobs run once and notify success.",
    "enabled_status": enabled_result.status,
    "disabled_status": disabled_result.status,
    "disabled_detail": disabled_result.detail,
    "locked_status": locked_result.status,
    "enabled_task_calls": calls["enabled"],
    "disabled_task_calls": calls["disabled"],
    "locked_task_calls": calls["locked"],
    "disabled_lock_created": disabled_lock_exists,
    "notification_events": events,
    "review_gate": "src/schedule job control must be explicit before data/model/dashboard/paper jobs are composed by schedule pipelines.",
}

(OUTPUT_DIR / "schedule_job_control_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(OUTPUT_DIR / "schedule_job_control_review.md").write_text(
    "# Schedule Job Control Review\n\n"
    "## 中文结论\n\n"
    f"- 做了什么：验证 enabled、disabled、locked 三类通用 schedule job 控制路径。\n"
    f"- 核心指标：enabled_status={enabled_result.status}；disabled_status={disabled_result.status}；locked_status={locked_result.status}；disabled_task_calls={calls['disabled']}。\n"
    "- regime 差异：本卡片不评估市场 regime；它保证不同调度状态下的任务执行差异可控。\n"
    "- 是否晋升：否；这是 src/schedule 稳定契约验证，不是策略晋升。\n"
    "- 风险：若 disabled job 仍执行，会导致数据更新、dashboard refresh 或 paper execution 在暂停状态下误运行。\n"
    "- 下一步：继续把定时触发的 calendar 判断、失败重试和通知摘要留在 schedule pipeline notebook 中审阅。\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.schedule_job_control.v1',
 'status': 'completed',
 'job_control_contract': 'disabled jobs skip before lock/task/notify; locked jobs skip existing concurrent work; enabled jobs run once and notify success.',
 'enabled_status': 'success',
 'disabled_status': 'skipped',
 'disabled_detail': 'disabled',
 'locked_status': 'locked',
 'enabled_task_calls': 1,
 'disabled_task_calls': 0,
 'locked_task_calls': 0,
 'disabled_lock_created': False,
 'notification_events': ['dashboard_export_enabled:success'],
 'review_gate': 'src/schedule job control must be explicit before data/model/dashboard/paper jobs are composed by schedule pipelines.'}

## 中文结论

- 做了什么：补充并验证 `src/schedule` 的通用 job 控制契约，覆盖 enabled、disabled 和 locked 三种调度状态。
- 核心指标：disabled job 预期返回 `skipped`，detail 为 `disabled`，task 调用次数为 0，且不创建 lock、不发送成功/失败通知；enabled job 仍返回 success；locked job 仍返回 locked。
- regime 差异：本 notebook 不分析市场 regime；这里的差异是调度状态差异，保证后续数据更新、模型刷新、dashboard export、paper execution 的开关行为稳定。
- 是否晋升：否；这是 schedule 框架契约补强，不是模型或策略晋升。
- 风险：如果 disabled 状态没有在通用入口拦截，暂停中的定时任务仍可能执行数据写入、通知或 paper/sim 操作。
- 下一步：继续把 calendar gate、retry 预算、通知摘要和 production health preflight 固化到 schedule pipeline 证据卡片。